# 12 — Phase 5: Streamlit 앱 v2 & 엣지 케이스 분석

> **목표:** Phase 4에서 완성한 3계층 XAI(`xai_explainer.py`)를 Streamlit v2 앱에 통합
>
> **산출물:**
> - `app/streamlit_app.py` (v2) — 3계층 XAI 통합 완전판
> - 엣지 케이스 정성 분석 리포트 (`reports/phase5/edge_case_analysis.png`)
> - AI 활용 로그 (`logs/prompt_engineering/phase5_log.md`)
>
> **Phase 5 체크리스트:**
> - [ ] Drive 마운트 + 경로 설정
> - [ ] xai_explainer.py 로드 확인
> - [ ] 엣지 케이스 정성 분석 (안경 / 어두운 조명 / Replay-Mask 혼동)
> - [ ] streamlit_app.py v2 작성 → Drive 저장
> - [ ] Colab에서 ngrok 터널로 실행 확인
> - [ ] 데모 시나리오 4건 스크린샷 저장

## Cell 0 — Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive 마운트 완료')

## Cell 1 — 경로 설정 & 라이브러리

In [ ]:
import os, sys, json
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib
from pathlib import Path

# ── 경로 설정 ──────────────────────────────────────────
BASE      = '/content/drive/MyDrive/face-anti-spoofing'
SRC_DIR   = f'{BASE}/src'
APP_DIR   = f'{BASE}/app'
CROP_DIR  = f'{BASE}/data/subset/cropped'
REPORT_DIR= f'{BASE}/reports/phase5'
RESULT_DIR= f'{BASE}/results/phase4'
LOG_DIR   = f'{BASE}/logs/prompt_engineering'

for d in [REPORT_DIR, APP_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# ── xai_explainer 로드 ──────────────────────────────────
sys.path.insert(0, SRC_DIR)
try:
    from xai_explainer import explain
    print('✅ xai_explainer.py 로드 완료')
except ImportError as e:
    print(f'⚠️  xai_explainer.py 미발견 — Phase 4-E 완료 후 재실행: {e}')

# ── 한글 폰트 ──────────────────────────────────────────
try:
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    fm.fontManager.addfont(font_path)
    prop = fm.FontProperties(fname=font_path)
    matplotlib.rcParams['font.family'] = prop.get_name()
    matplotlib.rcParams['axes.unicode_minus'] = False
    print('✅ 한글 폰트 적용')
except:
    matplotlib.rcParams['axes.unicode_minus'] = False
    print('⚠️  NanumGothic 없음 — 영문 대체')

print(f'BASE: {BASE}')

## Cell 2 — 엣지 케이스 정성 분석

Phase 5 분석 대상:
1. **안경 케이스** — Grad-CAM이 안경 테두리에 집중하는지
2. **어두운 조명** — Illumination 라벨 기반 FAR/FRR 변화
3. **Replay-Mask 혼동** — 중간 발표 Confusion Matrix 38건 집중 분석

In [ ]:
# ── 엣지 케이스 이미지 수집 ──────────────────────────────
# LLaVA 캡션 DB에서 특정 키워드가 포함된 이미지 추출

caption_db_path = f'{RESULT_DIR}/llava_captions.json'

try:
    with open(caption_db_path) as f:
        caption_db = json.load(f)
    print(f'✅ 캡션 DB 로드: {len(caption_db)}건')
except FileNotFoundError:
    print('⚠️  llava_captions.json 미발견 — 더미 DB로 진행')
    caption_db = {}

# 엣지 케이스 키워드 필터링
EDGE_KEYWORDS = {
    'glasses': ['glass', 'spectacl', 'eyewear', '안경'],
    'dark':    ['dark', 'dim', 'shadow', 'low light', '어둡', '조명'],
    'replay':  ['moire', 'screen', 'pixel', 'digital', 'replay'],
}

edge_cases = {k: [] for k in EDGE_KEYWORDS}

for img_path, info in caption_db.items():
    caption = info.get('llava_caption', '').lower()
    for edge_type, keywords in EDGE_KEYWORDS.items():
        if any(kw in caption for kw in keywords):
            edge_cases[edge_type].append({'path': img_path, **info})

for k, v in edge_cases.items():
    print(f'  {k:<10}: {len(v):>4}건')

In [ ]:
# ── 엣지 케이스 XAI 분석 & 시각화 ───────────────────────
# 각 케이스별 대표 이미지 2장씩 explain() 실행

EDGE_SAMPLE = 2  # 케이스당 샘플 수
edge_results = {}

try:
    for edge_type, cases in edge_cases.items():
        samples = cases[:EDGE_SAMPLE]
        if not samples:
            # 캡션 DB 없을 경우 크롭 폴더에서 직접 추출
            cat_map = {'glasses': 'live', 'dark': 'replay', 'replay': 'replay'}
            cat = cat_map.get(edge_type, 'replay')
            imgs = sorted(Path(f'{CROP_DIR}/{cat}').glob('*.jpg'))[:EDGE_SAMPLE]
            samples = [{'path': str(p), 'spoof_type': cat, 'llava_caption': ''} for p in imgs]

        edge_results[edge_type] = []
        for s in samples:
            img_bgr = cv2.imread(s['path'])
            if img_bgr is None:
                continue
            result = explain(img_bgr)
            result['img_rgb'] = cv2.cvtColor(cv2.resize(img_bgr, (224, 224)), cv2.COLOR_BGR2RGB)
            result['caption_db'] = s.get('llava_caption', '(N/A)')
            edge_results[edge_type].append(result)

    print('✅ 엣지 케이스 XAI 분석 완료')
    for k, v in edge_results.items():
        print(f'  {k}: {len(v)}건')

except Exception as e:
    print(f'⚠️  explain() 미로드 상태 — Phase 4-E 완료 후 재실행: {e}')

In [ ]:
# ── 엣지 케이스 시각화 저장 ───────────────────────────────

EDGE_LABELS = {
    'glasses': '안경 케이스 (Glasses)',
    'dark':    '어두운 조명 (Dark Illumination)',
    'replay':  'Replay-Mask 혼동 케이스'
}

if edge_results and any(edge_results.values()):
    n_types   = len([k for k, v in edge_results.items() if v])
    n_samples = EDGE_SAMPLE

    fig, axes = plt.subplots(
        n_types * n_samples, 3,
        figsize=(15, 5 * n_types * n_samples)
    )
    fig.suptitle('Phase 5: Edge Case Analysis — 3-Layer XAI',
                 fontsize=14, fontweight='bold', y=1.01)

    row = 0
    for edge_type, results in edge_results.items():
        if not results:
            continue
        label = EDGE_LABELS.get(edge_type, edge_type)
        for r in results:
            ax1, ax2, ax3 = axes[row] if n_types * n_samples > 1 else axes

            # 원본
            ax1.imshow(r['img_rgb'])
            color = 'red' if r['verdict'] == 'FAKE' else 'green'
            ax1.set_title(f'[{label}]\n{r["verdict"]} ({r["spoof_prob"]:.1%})',
                          fontsize=9, color=color)
            ax1.axis('off')

            # Grad-CAM
            ax2.imshow(r['heatmap_overlay'])
            ax2.set_title(f'Layer 1: Grad-CAM\nType: {r["spoof_type_name"]}', fontsize=9)
            ax2.axis('off')

            # 수치 + 캡션
            ax3.axis('off')
            summary = (
                f"Laplacian : {r['anchor_stats']['laplacian']}\n"
                f"FFT High  : {r['anchor_stats']['fft_high']}\n"
                f"Interp    : {r['anchor_interp']}\n\n"
                f"[XAI Text]\n{r['xai_text'][:150]}\n\n"
                f"[Caption DB]\n{r['caption_db'][:100]}"
            )
            ax3.text(0.03, 0.97, summary, transform=ax3.transAxes,
                     fontsize=7.5, verticalalignment='top',
                     bbox=dict(boxstyle='round', facecolor='#FFF9E6', alpha=0.85))
            ax3.set_title('Layer 2+3: Anchoring + Caption', fontsize=9)
            row += 1

    plt.tight_layout()
    out_path = f'{REPORT_DIR}/edge_case_analysis.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ 저장: {out_path}')
else:
    print('⚠️  edge_results 비어있음 — Cell 2 재실행 필요')

## Cell 3 — Streamlit 앱 v2 코드 작성

레이아웃 설계:
```
┌─────────────────────────────────────────────────────────┐
│  🛡️ 페이스페이 위조 공격 방어 시스템 v2                   │
│  [사이드바] 모델 설정 / FAR 대시보드                       │
├─────────────────────────────────────────────────────────┤
│  [업로드 영역]  →  [판정 결과 배너 REAL/FAKE]              │
├──────────────┬──────────────┬────────────────────────────│
│ Layer 1      │ Layer 2      │ Layer 3                    │
│ Grad-CAM     │ 수치 앵커링   │ 자연어 설명                 │
│ 히트맵        │ FFT/Laplacian│ LLaVA Caption              │
└──────────────┴──────────────┴────────────────────────────┘
  [하단] 공격 유형별 FAR 대시보드 (bar chart)
```

In [ ]:
# ── streamlit_app.py v2 작성 ─────────────────────────────

APP_CODE = '''
"""
app/streamlit_app.py  —  Phase 5 v2
🛡️ 페이스페이 위조 공격 방어 시스템 (3계층 XAI)

실행:
    streamlit run app/streamlit_app.py
Colab (ngrok):
    !pip install pyngrok -q
    !ngrok authtoken <YOUR_TOKEN>
    !streamlit run app/streamlit_app.py &
    from pyngrok import ngrok; print(ngrok.connect(8501))
"""

import sys, os, json
import numpy as np
import cv2
import streamlit as st
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

# ── 경로 설정 ──────────────────────────────────────────────
BASE     = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
SRC_DIR  = os.path.join(BASE, "src")
sys.path.insert(0, SRC_DIR)

# ── xai_explainer import ───────────────────────────────────
try:
    from xai_explainer import explain
    XAI_READY = True
except ImportError:
    XAI_READY = False

# ── FAR 실측 데이터 (Phase 4 결과 기반) ────────────────────
FAR_DATA = {
    "Print":  {"far": None, "target": 5.0,  "color": "#E74C3C"},
    "Replay": {"far": None, "target": 10.0, "color": "#F39C12"},
    "Mask":   {"far": None, "target": 8.0,  "color": "#9B59B6"},
}

def load_far_results():
    """Phase 4 FAR 분석 결과 로드."""
    far_path = os.path.join(BASE, "results", "phase4", "far_analysis.json")
    try:
        with open(far_path) as f:
            data = json.load(f)
        for cat in FAR_DATA:
            key = cat.lower() + "_far"
            if key in data:
                FAR_DATA[cat]["far"] = round(data[key] * 100, 2)
    except FileNotFoundError:
        pass  # 결과 파일 없으면 N/A 표시

load_far_results()

# ── Streamlit 페이지 설정 ─────────────────────────────────
st.set_page_config(
    page_title="FAS — 위조 공격 방어 시스템",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ── 사이드바 ───────────────────────────────────────────────
with st.sidebar:
    st.title("🛡️ FAS v2")
    st.caption("단국대학교 AI Security & Application")
    st.markdown("---")

    st.subheader("⚙️ 모델 설정")
    threshold = st.slider(
        "판정 임계값 (Spoof Prob > X → FAKE)",
        min_value=0.3, max_value=0.9,
        value=0.5, step=0.05
    )
    show_raw = st.checkbox("수치 상세 표시", value=True)

    st.markdown("---")
    st.subheader("📊 공격 유형별 FAR")

    for cat, info in FAR_DATA.items():
        far_val = info["far"]
        target  = info["target"]
        if far_val is not None:
            status = "✅" if far_val <= target else "⚠️"
            st.metric(
                label=f"{status} {cat}",
                value=f"{far_val:.1f}%",
                delta=f"목표 < {target}%",
                delta_color="inverse"
            )
        else:
            st.metric(
                label=f"⏳ {cat}",
                value="N/A",
                delta=f"목표 < {target}%"
            )

    st.markdown("---")
    st.caption("Phase 4 FAR 분석 결과 자동 반영")

# ── 메인 영역 ─────────────────────────────────────────────
st.title("🛡️ 페이스페이 위조 공격 방어 시스템")
st.caption("MobileNetV2 멀티태스크 + 3계층 XAI (Grad-CAM · 수치 앵커링 · 자연어 설명)")

if not XAI_READY:
    st.error(
        "⚠️ `src/xai_explainer.py`를 찾을 수 없습니다. "
        "Phase 4-E를 완료하고 파일을 저장한 뒤 앱을 재시작하세요."
    )
    st.stop()

# ── 이미지 업로드 ──────────────────────────────────────────
st.markdown("### 📷 이미지 업로드")
uploaded = st.file_uploader(
    "얼굴 이미지를 업로드하세요 (JPG / PNG / JPEG)",
    type=["jpg", "jpeg", "png"]
)

col_demo1, col_demo2 = st.columns([1, 4])
with col_demo1:
    demo_mode = st.checkbox("데모 모드 (샘플 이미지 사용)")

if demo_mode and not uploaded:
    demo_cats = ["live", "print", "replay", "mask"]
    demo_cat  = st.selectbox("샘플 카테고리 선택", demo_cats)
    demo_dir  = os.path.join(BASE, "data", "subset", "cropped", demo_cat)
    demo_imgs = sorted(Path(demo_dir).glob("*.jpg")) if os.path.isdir(demo_dir) else []
    if demo_imgs:
        demo_path = str(demo_imgs[0])
        img_bgr   = cv2.imread(demo_path)
        st.info(f"샘플 이미지: `{demo_cat}`")
    else:
        st.warning("샘플 이미지 없음 — 직접 업로드하세요")
        img_bgr = None
elif uploaded:
    file_bytes = np.frombuffer(uploaded.read(), np.uint8)
    img_bgr    = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
else:
    img_bgr = None

# ── 분석 실행 ─────────────────────────────────────────────
if img_bgr is not None:
    with st.spinner("🔍 분석 중..."):
        try:
            result = explain(img_bgr, threshold=threshold)
        except Exception as e:
            st.error(f"분석 오류: {e}")
            st.stop()

    # ── 판정 배너 ────────────────────────────────────────────
    verdict     = result["verdict"]
    spoof_prob  = result["spoof_prob"]
    spoof_type  = result["spoof_type_name"]

    if verdict == "FAKE":
        st.error(f"🚨 **FAKE** — 위조 공격 탐지됨 (신뢰도 {spoof_prob:.1%}) | 유형: **{spoof_type}**")
    else:
        st.success(f"✅ **REAL** — 실제 얼굴로 판정됨 (신뢰도 {1 - spoof_prob:.1%})")

    # ── 3단 레이아웃 ─────────────────────────────────────────
    col1, col2, col3 = st.columns(3)

    # Layer 1: Grad-CAM
    with col1:
        st.markdown("#### 🔥 Layer 1: Grad-CAM")
        st.caption("모델이 주목한 얼굴 영역")
        st.image(
            result["heatmap_overlay"],
            use_column_width=True,
            caption=f"Spoof Prob: {spoof_prob:.1%} | Type: {spoof_type}"
        )
        # 원본 비교
        img_rgb = cv2.cvtColor(cv2.resize(img_bgr, (224, 224)), cv2.COLOR_BGR2RGB)
        st.image(img_rgb, use_column_width=True, caption="원본 이미지")

    # Layer 2: 수치 앵커링
    with col2:
        st.markdown("#### 📐 Layer 2: 수치 앵커링")
        st.caption("Grad-CAM 활성 영역의 FFT / Laplacian 수치")

        anchor = result["anchor_stats"]
        lap_val = anchor.get("laplacian", "N/A")
        fft_val = anchor.get("fft_high",  "N/A")

        # 참조 기준값 (Phase 3 실측)
        REF = {
            "live":   {"lap": 383, "fft": 1134},
            "print":  {"lap": 318, "fft": 1042},
            "replay": {"lap": 319, "fft": 944},
            "mask":   {"lap": 480, "fft": 1134},
        }

        # 수치 메트릭 표시
        m1, m2 = st.columns(2)
        m1.metric("Laplacian", f"{lap_val:.0f}" if isinstance(lap_val, float) else lap_val,
                  help="활성화 영역 선명도 (낮을수록 블러 → Replay 가능성)")
        m2.metric("FFT High-Freq", f"{fft_val:.0f}" if isinstance(fft_val, float) else fft_val,
                  help="활성화 영역 고주파 에너지 (낮을수록 모아레 패턴)")

        st.markdown("")
        st.markdown("**📋 참조 기준값 (전체 평균)**")
        ref_table = "| 유형 | Laplacian | FFT |\n|---|---|---|\n"
        for cat, vals in REF.items():
            ref_table += f"| {cat.capitalize()} | {vals['lap']} | {vals['fft']} |\n"
        st.markdown(ref_table)

        if show_raw:
            with st.expander("수치 상세"):
                st.json(anchor)

        # Laplacian 막대 차트
        if isinstance(lap_val, (int, float)):
            fig_bar, ax = plt.subplots(figsize=(4, 2.5))
            cats = list(REF.keys()) + ["입력 이미지"]
            vals = [REF[c]["lap"] for c in REF] + [lap_val]
            colors = ["#95A5A6"] * 4 + ["#E74C3C" if verdict == "FAKE" else "#27AE60"]
            ax.barh(cats, vals, color=colors)
            ax.set_xlabel("Laplacian")
            ax.set_title("Laplacian 비교", fontsize=9)
            plt.tight_layout()
            st.pyplot(fig_bar)
            plt.close(fig_bar)

    # Layer 3: 자연어 설명
    with col3:
        st.markdown("#### 💬 Layer 3: 자연어 설명")
        st.caption("XAI 통합 설명 + LLaVA 캡션")

        # XAI 통합 텍스트
        xai_text = result.get("xai_text", "설명 생성 실패")
        if verdict == "FAKE":
            st.warning(f"🔍 **XAI 판정 근거**\n\n{xai_text}")
        else:
            st.info(f"🔍 **XAI 판정 근거**\n\n{xai_text}")

        # LLaVA 캡션
        llava_caption = result.get("llava_caption", None)
        st.markdown("**🤖 LLaVA 시각적 분석:**")
        if llava_caption and llava_caption != "(none)":
            st.success(llava_caption)
        else:
            st.caption("(LLaVA 캡션 DB에 해당 이미지 없음)")

        # Spoof Type 상세
        st.markdown("")
        st.markdown("**📌 공격 유형 해설:**")
        TYPE_DESC = {
            "Live":      "✅ 실제 얼굴. 위조 패턴 없음.",
            "Print":     "🖨️ 인쇄 공격. 종이/화면의 평면적 반사광 패턴 검출.",
            "Replay":    "📺 재촬영 공격. 화면 고주파 모아레 패턴 검출.",
            "3D Mask":   "😷 3D 마스크 공격. 경계선 및 텍스처 불연속 검출.",
            "Unknown":   "❓ 유형 불명. 복합 공격 또는 저품질 이미지.",
        }
        desc = TYPE_DESC.get(spoof_type, TYPE_DESC["Unknown"])
        st.markdown(desc)

    # ── FAR 대시보드 (하단) ───────────────────────────────────
    st.markdown("---")
    st.markdown("### 📊 공격 유형별 FAR 대시보드")
    st.caption("Phase 4 FAR 분석 결과 | 목표치 대비 달성 여부")

    has_far = any(v["far"] is not None for v in FAR_DATA.values())

    if has_far:
        fig_far, ax_far = plt.subplots(figsize=(8, 3))
        cats    = list(FAR_DATA.keys())
        actuals = [FAR_DATA[c]["far"] or 0 for c in cats]
        targets = [FAR_DATA[c]["target"] for c in cats]
        colors  = [
            "#27AE60" if (FAR_DATA[c]["far"] or 999) <= FAR_DATA[c]["target"]
            else "#E74C3C"
            for c in cats
        ]
        x = np.arange(len(cats))
        bars = ax_far.bar(x - 0.2, actuals, 0.35, label="실측 FAR", color=colors, alpha=0.85)
        ax_far.bar(x + 0.2, targets, 0.35, label="목표 FAR", color="#BDC3C7", alpha=0.7)
        ax_far.set_xticks(x)
        ax_far.set_xticklabels(cats, fontsize=11)
        ax_far.set_ylabel("FAR (%)")
        ax_far.set_title("공격 유형별 FAR: 실측 vs 목표", fontsize=12)
        ax_far.legend()
        for bar, val in zip(bars, actuals):
            ax_far.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                        f"{val:.1f}%", ha="center", fontsize=9)
        plt.tight_layout()
        st.pyplot(fig_far)
        plt.close(fig_far)
    else:
        st.info(
            "FAR 분석 결과 파일(`results/phase4/far_analysis.json`)이 없습니다. "
            "Phase 4-C 완료 후 자동으로 표시됩니다."
        )

else:
    # 초기 안내 화면
    st.markdown("---")
    st.info(
        "📷 **이미지를 업로드하거나 데모 모드를 켜세요.**\n\n"
        "**지원 공격 유형:** Print · Replay · 3D Mask · Live\n\n"
        "**3계층 XAI:**\n"
        "- Layer 1: logit 기반 Grad-CAM (어디를 봤는가)\n"
        "- Layer 2: 활성 영역 수치 앵커링 (얼마나 강한 신호인가)\n"
        "- Layer 3: LLaVA 자연어 설명 (왜 그렇게 판단했는가)"
    )
    col_a, col_b, col_c = st.columns(3)
    col_a.metric("Binary Accuracy", "96%",   "목표 >95% ✅")
    col_b.metric("Spoof Type Acc", "80%",    "목표 >80% ✅")
    col_c.metric("데이터셋",        "6,000장", "CelebA-Spoof")
'''

# Drive에 저장
app_path = f'{APP_DIR}/streamlit_app.py'
with open(app_path, 'w', encoding='utf-8') as f:
    f.write(APP_CODE.strip())

print(f'✅ streamlit_app.py v2 저장 완료: {app_path}')
print(f'   파일 크기: {os.path.getsize(app_path):,} bytes')

## Cell 4 — Colab에서 Streamlit 실행 (ngrok 터널)

> ⚠️ `YOUR_NGROK_TOKEN`을 https://dashboard.ngrok.com 에서 발급받아 교체하세요.

In [ ]:
# ── 의존성 설치 ────────────────────────────────────────────
!pip install streamlit pyngrok -q

# xai_explainer 의존성 확인
!pip install tensorflow opencv-python-headless -q 2>/dev/null || true

In [ ]:
# ── ngrok 토큰 설정 ───────────────────────────────────────
NGROK_TOKEN = 'YOUR_NGROK_TOKEN'  # ← 여기에 토큰 입력

!ngrok authtoken {NGROK_TOKEN}

# ── Streamlit 백그라운드 실행 ─────────────────────────────
import subprocess, time

proc = subprocess.Popen(
    ['streamlit', 'run', f'{APP_DIR}/streamlit_app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(4)  # 기동 대기

# ── ngrok 터널 ────────────────────────────────────────────
from pyngrok import ngrok
public_url = ngrok.connect(8501)
print(f'\n🌐 Streamlit 앱 URL: {public_url}')
print('   위 URL을 브라우저에서 열면 앱을 확인할 수 있습니다')

## Cell 5 — AI 활용 로그 작성

In [ ]:
# ── Phase 5 AI 활용 로그 생성 ────────────────────────────

LOG_CONTENT = '''# Phase 5 AI 활용 로그

## 작성일: 2026-05-21
## 단계: Phase 5 — Streamlit 앱 v2 & 엣지 케이스 분석

---

## Claude 프롬프트 엔지니어링 로그

### 1. Streamlit 앱 v2 설계
**프롬프트 의도:** Phase 4 xai_explainer.py를 통합한 완전판 앱 구조 설계
**핵심 요청:**
- 3단 레이아웃 (Grad-CAM / 수치 앵커링 / 자연어 설명)
- 사이드바에 FAR 대시보드 + 임계값 슬라이더
- 데모 모드 (샘플 이미지 자동 선택)
- FAR 분석 결과 JSON 자동 로드

**Claude 제안 핵심:**
- `xai_explainer.explain()` 호출 구조 유지
- Laplacian 비교 bar chart — 입력 이미지 vs 4카테고리 기준값
- FAR 대시보드 — 실측 vs 목표값 grouped bar
- `XAI_READY` 플래그로 Phase 4 미완료 시 graceful degradation

**반영 여부:** ✅ 전체 반영

---

### 2. 엣지 케이스 분석 설계
**프롬프트 의도:** 안경 / 어두운 조명 / Replay-Mask 혼동 케이스 정성 분석 방법
**Claude 제안:**
- LLaVA 캡션 DB에서 키워드 필터링으로 케이스 수집
- 캡션 DB 없을 경우 crop 폴더에서 직접 fallback
- 3케이스 × 2샘플 × 3컬럼 시각화 그리드

**반영 여부:** ✅ 전체 반영

---

## GitHub Copilot 활용
- streamlit_app.py 반복 컬럼 레이아웃 코드 자동완성
- FAR_DATA 딕셔너리 구조 → 루프 코드 자동완성

---

## 주요 설계 결정 기록

| 결정 사항 | 선택 | 이유 |
|-----------|------|------|
| FAR 대시보드 위치 | 하단 전체 폭 | 3계층 XAI가 핵심 — FAR는 보조 정보 |
| ngrok vs localhost | ngrok | Colab 환경에서 외부 접근 필요 |
| 임계값 슬라이더 | 사이드바 0.3~0.9 | 데모 시 실시간 변경 시연 가능 |
| LLaVA 캡션 없을 때 | graceful fallback | Phase 4 미완료 상태도 데모 가능하도록 |

---

## 다음 단계 (Phase 6)
- [ ] 데모 영상 녹화 (4케이스 × 3계층 XAI 출력)
- [ ] 슬라이드 최종본 — Phase 4~5 개선사항 반영
- [ ] GitHub README.md 최종 업데이트
'''

log_path = f'{LOG_DIR}/phase5_log.md'
with open(log_path, 'w', encoding='utf-8') as f:
    f.write(LOG_CONTENT)

print(f'✅ AI 활용 로그 저장: {log_path}')

## ✅ Phase 5 완료 체크리스트

| 항목 | 상태 |
|------|------|
| Drive 마운트 + 경로 설정 | ⬜ |
| xai_explainer.py 로드 확인 | ⬜ |
| 엣지 케이스 수집 (캡션 DB 키워드 필터링) | ⬜ |
| 엣지 케이스 XAI 분석 (explain() 실행) | ⬜ |
| 엣지 케이스 시각화 저장 (`edge_case_analysis.png`) | ⬜ |
| `app/streamlit_app.py` v2 작성 & Drive 저장 | ⬜ |
| ngrok 터널 실행 확인 | ⬜ |
| 데모 시나리오 4건 스크린샷 저장 | ⬜ |
| AI 활용 로그 작성 (`phase5_log.md`) | ⬜ |

---

### 다음 단계 — Phase 6: 최종 발표 준비
- **슬라이드 최종본**: Phase 4~5 개선사항 반영 (logit Grad-CAM / 3계층 XAI / FAR 실측)
- **라이브 데모**: 4케이스 시나리오 준비 + 백업 영상
- **예상 Q&A 8개** 답변 초안 완성
- **GitHub 최종 정리**: `results/final/` FAR 분석 + 캡션 데이터셋